# Chapter 11: Reshaping Data: Pivoting, Stacking, and Aggregation

Companion notebook for *Master Data Cleaning: A Hands-On Beginner's Guide with Python, pandas & NumPy*.

Run the cells from top to bottom (**Kernel → Restart Kernel and Run All Cells**). The data files this chapter uses are in the same folder as this notebook.

## 11.1 Wide to Long Format

### `df.stack()`

In [1]:
import numpy as np
import pandas as pd

quarterly = pd.DataFrame({
    'product': ['Laptop', 'Phone'],
    'Q1': [100, 150],
    'Q2': [120, 160],
    'Q3': [110, 155],
    'Q4': [130, 170],
}).set_index('product')
quarterly

,Q1,Q2,Q3,Q4
product,,,,
Laptop,100,120,110,130
Phone,150,160,155,170


In [2]:
quarterly_stacked = quarterly.stack()
quarterly_stacked

product    
Laptop   Q1    100
         Q2    120
         Q3    110
         Q4    130
Phone    Q1    150
         Q2    160
         Q3    155
         Q4    170
dtype: int64

In [3]:
quarterly_long = quarterly_stacked.reset_index()
quarterly_long.columns = ['product', 'quarter', 'units']
quarterly_long

,product,quarter,units
0,Laptop,Q1,100
1,Laptop,Q2,120
2,Laptop,Q3,110
3,Laptop,Q4,130
4,Phone,Q1,150
5,Phone,Q2,160
6,Phone,Q3,155
7,Phone,Q4,170


### `pd.melt()`

In [4]:
employee_hours = pd.DataFrame({
    'employee_id': ['E001', 'E002'],
    'department': ['Sales', 'Marketing'],
    'hours_2021': [2080, 1950],
    'hours_2022': [2000, 2050],
    'hours_2023': [2100, 2080],
})
employee_hours

,employee_id,department,hours_2021,hours_2022,hours_2023
0,E001,Sales,2080,2000,2100
1,E002,Marketing,1950,2050,2080


In [5]:
hours_long = pd.melt(
    employee_hours,
    id_vars=['employee_id', 'department'],
    value_vars=['hours_2021', 'hours_2022', 'hours_2023'],
    var_name='year',
    value_name='hours',
)
hours_long

,employee_id,department,year,hours
0,E001,Sales,hours_2021,2080
1,E002,Marketing,hours_2021,1950
2,E001,Sales,hours_2022,2000
3,E002,Marketing,hours_2022,2050
4,E001,Sales,hours_2023,2100
5,E002,Marketing,hours_2023,2080


In [6]:
hours_long['year'] = (
    hours_long['year'].str.replace('hours_', '').astype(int)
)
hours_long

,employee_id,department,year,hours
0,E001,Sales,2021,2080
1,E002,Marketing,2021,1950
2,E001,Sales,2022,2000
3,E002,Marketing,2022,2050
4,E001,Sales,2023,2100
5,E002,Marketing,2023,2080


### `pd.wide_to_long()`

In [7]:
student_scores = pd.DataFrame({
    'student_id': [1, 2],
    'name': ['Aisha', 'Ben'],
    'math_2021': [85, 90],
    'math_2022': [88, 92],
    'english_2021': [82, 88],
    'english_2022': [85, 90],
})
student_scores

,student_id,name,math_2021,math_2022,english_2021,english_2022
0,1,Aisha,85,88,82,85
1,2,Ben,90,92,88,90


In [8]:
scores_long = pd.wide_to_long(
    student_scores,
    stubnames=['math', 'english'],
    i='student_id',
    j='year',
    sep='_',
).reset_index()
scores_long

,student_id,year,name,math,english
0,1,2021,Aisha,85,82
1,2,2021,Ben,90,88
2,1,2022,Aisha,88,85
3,2,2022,Ben,92,90


## 11.2 Long to Wide Format

### `df.pivot()`

In [9]:
hours_wide = hours_long.pivot(
    index='employee_id', columns='year', values='hours'
)
hours_wide

year,2021,2022,2023
employee_id,,,
E001,2080,2000,2100
E002,1950,2050,2080


In [10]:
hours_wide.reset_index()

year,employee_id,2021,2022,2023
0,E001,2080,2000,2100
1,E002,1950,2050,2080


In [11]:
hours_long.pivot(
    index=['employee_id', 'department'],
    columns='year',
    values='hours',
)

,year,2021,2022,2023
employee_id,department,,,
E001,Sales,2080,2000,2100
E002,Marketing,1950,2050,2080


### `df.unstack()`

In [12]:
scores_indexed = scores_long.set_index(['student_id', 'year'])
scores_indexed

,,name,math,english
student_id,year,,,
1,2021,Aisha,85,82
2,2021,Ben,90,88
1,2022,Aisha,88,85
2,2022,Ben,92,90


In [13]:
scores_indexed['math'].unstack()

year,2021,2022
student_id,,
1,85,88
2,90,92


In [14]:
scores_indexed['math'].unstack(level='student_id')

student_id,1,2
year,,
2021,85,90
2022,88,92


In [15]:
scores_wide = scores_indexed[['math', 'english']].unstack()
scores_wide

math      english     
year       2021 2022    2021 2022
student_id                       
1            85   88      82   85
2            90   92      88   90

In [16]:
scores_flat = scores_wide.copy()
scores_flat.columns = [
    f'{subject}_{year}' for subject, year in scores_flat.columns
]
scores_flat.reset_index()

,student_id,math_2021,math_2022,english_2021,english_2022
0,1,85,88,82,85
1,2,90,92,88,90


## 11.3 Working with MultiIndex Levels

### `pd.MultiIndex.from_product()`

In [17]:
sales_index = pd.MultiIndex.from_product(
    [['East', 'West'], ['Q1', 'Q2'], [2023, 2024]],
    names=['region', 'quarter', 'year'],
)
regional = pd.DataFrame({
    'revenue': [100, 110, 120, 130, 95, 105, 110, 125],
    'units': [50, 55, 60, 65, 48, 52, 55, 62],
}, index=sales_index)
regional

revenue  units
region quarter year                
East   Q1      2023      100     50
               2024      110     55
       Q2      2023      120     60
               2024      130     65
West   Q1      2023       95     48
               2024      105     52
       Q2      2023      110     55
               2024      125     62

In [18]:
regional.index.levels

FrozenList([['East', 'West'], ['Q1', 'Q2'], [2023, 2024]])

### Selecting with `df.loc[]`

In [19]:
regional.loc['East']

revenue  units
quarter year                
Q1      2023      100     50
        2024      110     55
Q2      2023      120     60
        2024      130     65

In [20]:
regional.loc[('East', 'Q1')]

,revenue,units
year,,
2023,100,50
2024,110,55


In [21]:
regional.loc[('East', 'Q1', 2023)]

revenue    100
units       50
Name: (East, Q1, 2023), dtype: int64

### `pd.IndexSlice`

In [22]:
idx = pd.IndexSlice
regional.loc[idx[:, 'Q1', :], :]

revenue  units
region quarter year                
East   Q1      2023      100     50
               2024      110     55
West   Q1      2023       95     48
               2024      105     52

In [23]:
regional.loc[idx[:, ['Q1', 'Q2'], 2024], :]

revenue  units
region quarter year                
East   Q1      2024      110     55
       Q2      2024      130     65
West   Q1      2024      105     52
       Q2      2024      125     62

### `df.xs()`

In [24]:
regional.xs(2024, level='year')

revenue  units
region quarter                
East   Q1           110     55
       Q2           130     65
West   Q1           105     52
       Q2           125     62

### `df.swaplevel()` and `df.sort_index()`

In [25]:
regional.swaplevel('region', 'year').head(4)

,,,revenue,units
year,quarter,region,,
2023,Q1,East,100,50
2024,Q1,East,110,55
2023,Q2,East,120,60
2024,Q2,East,130,65


In [26]:
regional.sort_index(level='year', ascending=False).head(4)

revenue  units
region quarter year                
West   Q2      2024      125     62
       Q1      2024      105     52
East   Q2      2024      130     65
       Q1      2024      110     55

### Aggregating by level

In [27]:
regional.groupby(level='region').sum()

,revenue,units
region,,
East,460,230
West,435,217


In [28]:
regional.groupby(level=['region', 'quarter']).sum()

revenue  units
region quarter                
East   Q1           210    105
       Q2           250    125
West   Q1           200    100
       Q2           235    117

### MultiIndex columns

In [29]:
regional_by_year = regional.unstack(level='year')
regional_by_year

revenue      units     
year              2023 2024  2023 2024
region quarter                        
East   Q1          100  110    50   55
       Q2          120  130    60   65
West   Q1           95  105    48   52
       Q2          110  125    55   62

In [30]:
regional_by_year.xs(2024, level='year', axis=1)

revenue  units
region quarter                
East   Q1           110     55
       Q2           130     65
West   Q1           105     52
       Q2           125     62

In [31]:
regional_flat = regional_by_year.copy()
regional_flat.columns = [
    f'{measure}_{year}' for measure, year in regional_flat.columns
]
regional_flat

revenue_2023  revenue_2024  units_2023  units_2024
region quarter                                                    
East   Q1                100           110          50          55
       Q2                120           130          60          65
West   Q1                 95           105          48          52
       Q2                110           125          55          62

## 11.4 Split-Apply-Combine with `groupby()`

In [32]:
sales = pd.read_csv('sales.csv')
sales

,order_id,region,product,quarter,payment,revenue,units
0,1001,East,Laptop,Q1,Card,4200.0,3
1,1002,East,Phone,Q1,Cash,2600.0,4
2,1003,East,Laptop,Q2,Card,5600.0,4
3,1004,West,Phone,Q1,Card,1950.0,3
4,1005,West,Laptop,Q2,Cash,2800.0,2
5,1006,West,Phone,Q2,Card,NaN,5
6,1007,North,Laptop,Q1,Card,7000.0,5
7,1008,North,Phone,Q2,Cash,3250.0,5
8,1009,North,Phone,Q2,Card,1300.0,2
9,1010,East,Phone,Q2,Card,1950.0,3


### `df.groupby()`

In [33]:
by_region = sales.groupby('region')
print(by_region.ngroups)
print(list(by_region.groups))

3
['East', 'North', 'West']


In [34]:
sales.groupby('region')['revenue'].sum()

region
East     14350.0
North    11550.0
West      4750.0
Name: revenue, dtype: float64

In [35]:
sales.groupby('region')[['revenue', 'units']].sum()

,revenue,units
region,,
East,14350.0,14
North,11550.0,12
West,4750.0,10


### Grouping by several columns

In [36]:
sales.groupby(['region', 'quarter'])['revenue'].sum()

region  quarter
East    Q1         6800.0
        Q2         7550.0
North   Q1         7000.0
        Q2         4550.0
West    Q1         1950.0
        Q2         2800.0
Name: revenue, dtype: float64

In [37]:
sales.groupby(['region', 'quarter'])['revenue'].sum().unstack()

quarter,Q1,Q2
region,,
East,6800.0,7550.0
North,7000.0,4550.0
West,1950.0,2800.0


### Keeping the keys as columns

In [38]:
sales.groupby('region', as_index=False)[['revenue', 'units']].sum()

,region,revenue,units
0,East,14350.0,14
1,North,11550.0,12
2,West,4750.0,10


### Grouping by an array or an index level

In [39]:
large_order = sales['units'] >= 3
sales.groupby(large_order)['revenue'].mean()

units
False    2050.000000
True     3792.857143
Name: revenue, dtype: float64

## 11.5 Aggregating Groups

### `count()` versus `size()`

In [40]:
sales.groupby('region')[['revenue', 'units']].count()

,revenue,units
region,,
East,4,4
North,3,3
West,2,3


In [41]:
sales.groupby('region').size()

region
East     4
North    3
West     3
dtype: int64

In [42]:
sales.groupby('region')['revenue'].mean()

region
East     3587.5
North    3850.0
West     2375.0
Name: revenue, dtype: float64

### `GroupBy.agg()`

In [43]:
sales.groupby('region')['revenue'].agg(['sum', 'mean', 'count'])

,sum,mean,count
region,,,
East,14350.0,3587.5,4
North,11550.0,3850.0,3
West,4750.0,2375.0,2


In [44]:
sales.groupby('region').agg({
    'revenue': ['sum', 'max'],
    'units': 'sum',
    'product': 'nunique',
})

revenue         units product
            sum     max   sum nunique
region                               
East    14350.0  5600.0    14       2
North   11550.0  7000.0    12       2
West     4750.0  2800.0    10       2

### Named aggregation

In [45]:
region_summary = sales.groupby('region').agg(
    orders=('order_id', 'count'),
    total_revenue=('revenue', 'sum'),
    avg_revenue=('revenue', 'mean'),
    total_units=('units', 'sum'),
)
region_summary

,orders,total_revenue,avg_revenue,total_units
region,,,,
East,4,14350.0,3587.5,14
North,3,11550.0,3850.0,12
West,3,4750.0,2375.0,10


### Custom aggregation functions

In [46]:
sales.groupby('region').agg(
    total_revenue=('revenue', 'sum'),
    revenue_range=('revenue', lambda s: s.max() - s.min()),
)

,total_revenue,revenue_range
region,,
East,14350.0,3650.0
North,11550.0,5700.0
West,4750.0,850.0


### `GroupBy.describe()`

In [47]:
sales.groupby('region')['units'].describe()

,count,mean,std,min,25%,50%,75%,max
region,,,,,,,,
East,4.0,3.500000,0.577350,3.0,3.0,3.5,4.0,4.0
North,3.0,4.000000,1.732051,2.0,3.5,5.0,5.0,5.0
West,3.0,3.333333,1.527525,2.0,2.5,3.0,4.0,5.0


## 11.6 Transform, Apply and Filter

In [48]:
staff = pd.DataFrame({
    'department': ['Sales', 'Sales', 'Sales', 'IT', 'IT', 'IT',
                   'HR', 'HR', 'HR'],
    'employee': ['Alya', 'Ben', 'Chen', 'Dev', 'Elena', 'Farid',
                 'Grace', 'Hana', 'Ivan'],
    'salary': [60000, 65000, 62000, 75000, 80000, 78000,
               55000, 58000, 56000],
})
staff

,department,employee,salary
0,Sales,Alya,60000
1,Sales,Ben,65000
2,Sales,Chen,62000
3,IT,Dev,75000
4,IT,Elena,80000
5,IT,Farid,78000
6,HR,Grace,55000
7,HR,Hana,58000
8,HR,Ivan,56000


### `GroupBy.transform()`

In [49]:
staff.groupby('department')['salary'].mean().round(2)

department
HR       56333.33
IT       77666.67
Sales    62333.33
Name: salary, dtype: float64

In [50]:
staff.groupby('department')['salary'].transform('mean').round(2)

0    62333.33
1    62333.33
2    62333.33
3    77666.67
4    77666.67
5    77666.67
6    56333.33
7    56333.33
8    56333.33
Name: salary, dtype: float64

In [51]:
dept_salary = staff.groupby('department')['salary']
staff['dept_avg'] = dept_salary.transform('mean').round(2)
staff['vs_avg'] = (
    staff['salary'] - dept_salary.transform('mean')
).round(2)
staff['z_score'] = (
    (staff['salary'] - dept_salary.transform('mean'))
    / dept_salary.transform('std')
).round(2)
staff

,department,employee,salary,dept_avg,vs_avg,z_score
0,Sales,Alya,60000,62333.33,-2333.33,-0.93
1,Sales,Ben,65000,62333.33,2666.67,1.06
2,Sales,Chen,62000,62333.33,-333.33,-0.13
3,IT,Dev,75000,77666.67,-2666.67,-1.06
4,IT,Elena,80000,77666.67,2333.33,0.93
5,IT,Farid,78000,77666.67,333.33,0.13
6,HR,Grace,55000,56333.33,-1333.33,-0.87
7,HR,Hana,58000,56333.33,1666.67,1.09
8,HR,Ivan,56000,56333.33,-333.33,-0.22


In [52]:
def pct_of_max(salaries):
    return (salaries / salaries.max() * 100).round(2)

staff['pct_of_max'] = dept_salary.transform(pct_of_max)
staff[['department', 'employee', 'salary', 'pct_of_max']]

,department,employee,salary,pct_of_max
0,Sales,Alya,60000,92.31
1,Sales,Ben,65000,100.00
2,Sales,Chen,62000,95.38
3,IT,Dev,75000,93.75
4,IT,Elena,80000,100.00
5,IT,Farid,78000,97.50
6,HR,Grace,55000,94.83
7,HR,Hana,58000,100.00
8,HR,Ivan,56000,96.55


### Filling missing values by group

In [53]:
region_mean = sales.groupby('region')['revenue'].transform('mean')
sales['revenue_filled'] = sales['revenue'].fillna(region_mean)
sales[['order_id', 'region', 'revenue', 'revenue_filled']]

,order_id,region,revenue,revenue_filled
0,1001,East,4200.0,4200.0
1,1002,East,2600.0,2600.0
2,1003,East,5600.0,5600.0
3,1004,West,1950.0,1950.0
4,1005,West,2800.0,2800.0
5,1006,West,NaN,2375.0
6,1007,North,7000.0,7000.0
7,1008,North,3250.0,3250.0
8,1009,North,1300.0,1300.0
9,1010,East,1950.0,1950.0


### `GroupBy.apply()`

In [54]:
def region_report(group):
    return pd.Series({
        'orders': len(group),
        'total_revenue': group['revenue'].sum(),
        'best_product': (
            group.groupby('product')['revenue'].sum().idxmax()
        ),
    })

sales.groupby('region').apply(region_report)

,orders,total_revenue,best_product
region,,,
East,4,14350.0,Laptop
North,3,11550.0,Laptop
West,3,4750.0,Laptop


In [55]:
sales.groupby('region').apply(
    lambda group: group.nlargest(2, 'revenue')
)[['order_id', 'product', 'revenue']]

order_id product  revenue
region                             
East   2      1003  Laptop   5600.0
       0      1001  Laptop   4200.0
North  6      1007  Laptop   7000.0
       7      1008   Phone   3250.0
West   4      1005  Laptop   2800.0
       3      1004   Phone   1950.0

In [56]:
(
    sales.sort_values('revenue', ascending=False)
    .groupby('region')
    .head(2)[['order_id', 'region', 'revenue']]
)

,order_id,region,revenue
6,1007,North,7000.0
2,1003,East,5600.0
0,1001,East,4200.0
7,1008,North,3250.0
4,1005,West,2800.0
3,1004,West,1950.0


### `GroupBy.filter()`

In [57]:
sales.groupby('region').filter(
    lambda group: group['revenue'].sum() > 12000
)[['order_id', 'region', 'revenue']]

,order_id,region,revenue
0,1001,East,4200.0
1,1002,East,2600.0
2,1003,East,5600.0
9,1010,East,1950.0


## 11.7 Pivot Tables and Cross-Tabulation

### `pd.pivot_table()`

In [58]:
pd.pivot_table(
    sales,
    values='revenue',
    index='region',
    columns='quarter',
    aggfunc='sum',
)

quarter,Q1,Q2
region,,
East,6800.0,7550.0
North,7000.0,4550.0
West,1950.0,2800.0


In [59]:
pd.pivot_table(
    sales,
    values='revenue',
    index='region',
    columns='quarter',
    aggfunc=['sum', 'count'],
)

sum         count   
quarter      Q1      Q2    Q1 Q2
region                          
East     6800.0  7550.0     2  2
North    7000.0  4550.0     1  2
West     1950.0  2800.0     1  1

In [60]:
pd.pivot_table(
    sales,
    values='units',
    index=['region', 'product'],
    columns='quarter',
    aggfunc='sum',
    fill_value=0,
)

quarter         Q1  Q2
region product        
East   Laptop    3   4
       Phone     4   3
North  Laptop    5   0
       Phone     0   7
West   Laptop    0   2
       Phone     3   5

In [61]:
pd.pivot_table(
    sales,
    values=['revenue', 'units'],
    index='region',
    columns='quarter',
    aggfunc='sum',
)

revenue         units   
quarter      Q1      Q2    Q1 Q2
region                          
East     6800.0  7550.0     7  7
North    7000.0  4550.0     5  7
West     1950.0  2800.0     3  7

### Adding totals with `margins`

In [62]:
pd.pivot_table(
    sales,
    values='revenue',
    index='region',
    columns='quarter',
    aggfunc='sum',
    margins=True,
    margins_name='Total',
)

quarter,Q1,Q2,Total
region,,,
East,6800.0,7550.0,14350.0
North,7000.0,4550.0,11550.0
West,1950.0,2800.0,4750.0
Total,15750.0,14900.0,30650.0


### `pivot()` versus `pivot_table()`

The next cell raises an error on purpose.

In [63]:
sales.pivot(index='region', columns='quarter', values='revenue')

ValueError: Index contains duplicate entries, cannot reshape

### `pd.crosstab()`

In [64]:
pd.crosstab(sales['region'], sales['payment'])

payment,Card,Cash
region,,
East,3,1
North,2,1
West,2,1


In [65]:
pd.crosstab(sales['region'], sales['payment'], margins=True)

payment,Card,Cash,All
region,,,
East,3,1,4
North,2,1,3
West,2,1,3
All,7,3,10


### Percentages with `normalize`

In [66]:
pd.crosstab(
    sales['region'], sales['payment'], normalize='index'
).round(2)

payment,Card,Cash
region,,
East,0.75,0.25
North,0.67,0.33
West,0.67,0.33


### Several row variables

In [67]:
pd.crosstab(
    [sales['region'], sales['product']],
    sales['payment'],
)

payment         Card  Cash
region product            
East   Laptop      2     0
       Phone       1     1
North  Laptop      1     0
       Phone       1     1
West   Laptop      0     1
       Phone       2     0